In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from MedSAM.segment_anything import sam_model_registry
from MedSAM.MedSAM_Inference import medsam_inference
from skimage import io, transform
from tqdm import tqdm

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
medsam_model = sam_model_registry["vit_b"](checkpoint="MedSAM/work_dir/MedSAM/medsam_vit_b.pth")
medsam_model = medsam_model.to(device)
medsam_model.eval()

/home/paulb/INF8225/Projet/MedSAM/segment_anything/build_sam.py:144: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f, map_location=torch.device('cpu'

Sam(
  (image_encoder): ImageEncoderViT(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): Linear(in_features=3072, out_features=768, bias=True)
          (act): GELU(approximate='none')
        )
      )
    )
    (neck): Sequential(
      (0): Conv2d(768, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): LayerNorm2d()
      (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (3): LayerNorm2d()
    )


In [3]:
def calculate_dice(mask_true, mask_pred):
    m_true = np.asarray(mask_true).astype(bool)
    m_pred = np.asarray(mask_pred).astype(bool)

    if m_true.sum() + m_pred.sum() == 0:
        return 1.0
    
    intersection = np.logical_and(m_true, m_pred).sum()
    return 2 * intersection / (m_true.sum() + m_pred.sum())

In [7]:
# --- Configuration des chemins MSD ---
base_dir = "data/MSD_pancreas"
output_folder = os.path.join(base_dir, "outputs_baseline")
os.makedirs(output_folder, exist_ok=True)
os.makedirs("data/results", exist_ok=True)

# Chargement du fichier maître COCO
with open(os.path.join(base_dir, "annotations.json")) as f:
    coco_data = json.load(f)

# Groupement des annotations par image_id pour un accès rapide
annotations_by_image = {}
for ann in coco_data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in annotations_by_image:
        annotations_by_image[img_id] = []
    annotations_by_image[img_id].append(ann)

dice_list = []

# Boucle unique sur les 1000 images via le JSON
for img_info in tqdm(coco_data["images"], desc="Segmentation MedSAM"):
    img_id = img_info["id"]
    
    # img_info["file_name"] contient déjà "train/images/pancreas_001_s33.png"
    img_rel_path = img_info["file_name"] 
    img_path = os.path.join(base_dir, img_rel_path)
    
    # Déduction automatique du chemin du masque
    mask_rel_path = img_rel_path.replace("/images/", "/masks/")
    mask_path = os.path.join(base_dir, mask_rel_path)

    # Si l'image n'a pas d'annotations, on passe
    if img_id not in annotations_by_image:
        continue

    img_np = io.imread(img_path)
    if len(img_np.shape) == 2:
        img_3c = np.repeat(img_np[:, :, None], 3, axis=-1)
    else:
        img_3c = img_np
    H, W, _ = img_3c.shape

    # Préparation MedSAM
    img_1024 = transform.resize(
        img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
    ).astype(np.uint8)
    img_1024 = (img_1024 - img_1024.min()) / np.clip(
        img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
    ) 
    img_1024_tensor = (
        torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device)
    )

    with torch.no_grad():
        image_embedding = medsam_model.image_encoder(img_1024_tensor)

    full_medsam_seg = np.zeros((H, W), dtype=np.uint8)

    # Itération sur toutes les boîtes (Tumeur) de cette image
    for ann in annotations_by_image[img_id]:
        if ann["category_id"] != 2:
            continue
        # Conversion du format COCO [x, y, w, h] vers [x1, y1, x2, y2]
        x, y, w, h = ann["bbox"]
        x_min, y_min = x, y
        x_max, y_max = x + w, y + h
    
        box_np = np.array([[x_min, y_min, x_max, y_max]]) 
        box_1024 = box_np / np.array([W, H, W, H]) * 1024

        medsam_seg = medsam_inference(medsam_model, image_embedding, box_1024, H, W)
        full_medsam_seg[medsam_seg > 0] = 1

    io.imsave(
        os.path.join(output_folder, "seg_" + os.path.basename(img_path)),
        (full_medsam_seg * 255).astype(np.uint8),
        check_contrast=False,
    )

    has_tumor = any(ann["category_id"] == 2 for ann in annotations_by_image[img_id])

    # Chargement du masque vérité terrain
    true_seg_raw = io.imread(mask_path)
    # On ne cible que les pixels de valeur 2 (Tumeur)
    true_seg = np.zeros_like(true_seg_raw, dtype=np.uint8)
    true_seg[true_seg_raw == 2] = 1

    dice_score = calculate_dice(true_seg, full_medsam_seg)

    dice_list.append({
        "image_name": os.path.basename(img_path),
        "split": img_info["split"],
        "has_tumor": has_tumor,
        "dice": dice_score
    })
    
df = pd.DataFrame(dice_list)
df.to_csv("data/results/dice_baseline_msd.csv", index=False)

Segmentation MedSAM:   0%|          | 0/1000 [00:00<?, ?it/s]

Segmentation MedSAM: 100%|██████████| 1000/1000 [17:56<00:00,  1.08s/it]


In [8]:
df = pd.read_csv("data/results/dice_baseline_msd.csv")
df.head()

,image_name,split,has_tumor,dice
0,pancreas_001_s39.png,train,True,0.830799
1,pancreas_001_s49.png,train,True,0.837775
2,pancreas_001_s53.png,train,True,0.801556
3,pancreas_001_s54.png,train,True,0.774510
4,pancreas_004_s37.png,train,True,0.728464


In [9]:
# Chargement du fichier CSV
df = pd.read_csv("data/results/dice_baseline_msd.csv")

# 1. Statistiques Globales (Toutes les images)
print("=== STATISTIQUES GLOBALES ===")
print(df['dice'].describe())
print("\n")

# 2. Statistiques sur les images AVEC tumeur uniquement (Le vrai score de détection)
df_tumor = df[df['has_tumor'] == True]
print("=== STATISTIQUES (AVEC TUMEUR UNIQUEMENT) ===")
print(df_tumor['dice'].describe())
print("\n")

# 3. Statistiques par Split (Optionnel, très utile pour le rapport)
print("=== DSC MOYEN PAR SPLIT (Tumeurs Uniquement) ===")
print(df_tumor.groupby('split')['dice'].mean())

=== STATISTIQUES GLOBALES ===
count    1000.000000
mean        0.822625
std         0.206653
min         0.000000
25%         0.682032
50%         0.895858
75%         1.000000
max         1.000000
Name: dice, dtype: float64


=== STATISTIQUES (AVEC TUMEUR UNIQUEMENT) ===
count    600.000000
mean       0.704375
std        0.190281
min        0.000000
25%        0.580459
50%        0.728834
75%        0.861631
max        0.980782
Name: dice, dtype: float64


=== DSC MOYEN PAR SPLIT (Tumeurs Uniquement) ===
split
test     0.659788
train    0.710523
val      0.699771
Name: dice, dtype: float64
